# Wayfinder Eval Pipeline — GPU Accelerated

Runs the Stage 4 evaluation experiments that are too slow on MPS:
1. **Eval Retrieval** (EXP-2.1): Nav vs Dense recall with coverage-aware metrics on 242K entity DB
2. **Eval Spreading** (EXP-2.2): Spreading activation benefit
3. **Anchor Gap Analysis** (EXP-0.2 re-run): Validate recall@16 on expanded DB

The bottleneck is encoding 242K premise names through MiniLM + GoalAnalyzer (~20 min on MPS, ~2 min on T4).

**Setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Clone repo and install deps
!git clone https://github.com/rohanvinaik/Wayfinder.git /content/wayfinder
%cd /content/wayfinder
!pip install -q torch numpy pyyaml sentence-transformers transformers

In [ ]:
# Verify GPU available
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Runtime → Change runtime type → T4 GPU')

In [ ]:
# Verify data files exist
import os
required = [
    'data/proof_network.db',
    'data/nav_eval.jsonl',
    'data/nav_train.jsonl',
    'models/NAV-002_step5000.pt',
    'configs/wayfinder.yaml',
]
for f in required:
    exists = os.path.exists(f)
    size = os.path.getsize(f) / 1e6 if exists else 0
    status = f'{size:.1f} MB' if exists else 'MISSING'
    print(f'  {"✓" if exists else "✗"} {f}: {status}')

# If data files are missing, they need to be uploaded or generated
# The large files (proof_network.db, nav_train.jsonl, checkpoint) 
# should be in the repo via git-lfs or need manual upload

## If data files are missing

The large data files may not be in git. Upload them manually:
- `data/proof_network.db` (~300MB) — the expanded 242K entity proof network
- `data/nav_eval.jsonl` (~68MB) — eval examples with domain-fixed labels
- `models/NAV-002_step5000.pt` (~163MB) — trained navigator checkpoint

Or regenerate from source:
```python
# 1. Extract entities (needs data/leandojo_mathlib.jsonl + data/leandojo_benchmark_4/corpus.jsonl)
!python scripts/extract_proof_network.py --input data/leandojo_mathlib.jsonl \
    --output data/proof_network_entities.jsonl \
    --corpus data/leandojo_benchmark_4/corpus.jsonl

# 2. Build DB
!python scripts/build_proof_network_db.py \
    --input data/proof_network_entities.jsonl \
    --output data/proof_network.db
```

In [ ]:
# Upload data files from Google Drive or local machine
# Option 1: Google Drive (recommended for large files)
from google.colab import drive
drive.mount('/content/drive')

# Copy files from Drive — adjust paths to match your Drive structure
import shutil, os

drive_base = '/content/drive/MyDrive/Wayfinder'  # <-- EDIT this path

file_map = {
    f'{drive_base}/proof_network.db': 'data/proof_network.db',
    f'{drive_base}/nav_eval.jsonl': 'data/nav_eval.jsonl',
    f'{drive_base}/nav_train.jsonl': 'data/nav_train.jsonl',
    f'{drive_base}/NAV-002_step5000.pt': 'models/NAV-002_step5000.pt',
}

for src, dst in file_map.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f'  Copying {os.path.basename(src)} ({os.path.getsize(src)/1e6:.0f} MB)...')
            shutil.copy2(src, dst)
        else:
            print(f'  ✓ {dst} already exists')
    else:
        print(f'  ✗ {src} not found on Drive')
        print(f'    Upload to Google Drive at: {src}')

# Option 2: Direct upload (for smaller files or if Drive not available)
# Uncomment below to upload interactively:
# from google.colab import files
# uploaded = files.upload()  # drag and drop files

---
## Experiment 1: Eval Retrieval (EXP-2.1)

Compares navigational retrieval vs dense cosine similarity baseline.
Reports both raw recall and **conditional recall** (only counting premises that exist in the DB).

**Gate:** nav recall@16 ≥ 80% of dense recall@16

In [ ]:
import os
os.chdir('/content/wayfinder')
os.environ['PYTHONPATH'] = '/content/wayfinder'

!PYTHONPATH=/content/wayfinder python scripts/eval_retrieval.py \
    --config configs/wayfinder.yaml \
    --checkpoint models/NAV-002_step5000.pt \
    --device cuda \
    --samples 500 \
    --output runs/eval_retrieval_v2_gpu.json

In [ ]:
# Display results
import json, os

result_path = 'runs/eval_retrieval_v2_gpu.json'
if not os.path.exists(result_path):
    print(f'ERROR: {result_path} not found. Check the eval_retrieval output above for errors.')
else:
    with open(result_path) as f:
        report = json.load(f)

    print('=== Universe Coverage ===')
    cov = report['universe_coverage']
    print(f"  Mean: {cov['mean']:.1%}")
    print(f"  Fully covered: {cov['fully_covered']}/{report['samples']}")
    print(f"  Zero covered: {cov['zero_covered']}/{report['samples']}")

    print('\n=== Raw Retrieval ===')
    for k in [1, 4, 8, 16]:
        nr = report['nav_retrieval'][f'recall@{k}']
        dr = report['dense_retrieval'][f'recall@{k}']
        print(f"  recall@{k}: nav={nr:.4f}  dense={dr:.4f}  delta={nr-dr:+.4f}")

    print('\n=== Conditional Retrieval (reachable premises only) ===')
    for k in [1, 4, 8, 16]:
        nr = report['nav_conditional_retrieval'][f'cond_recall@{k}']
        dr = report['dense_conditional_retrieval'][f'cond_recall@{k}']
        print(f"  cond_recall@{k}: nav={nr:.4f}  dense={dr:.4f}  delta={nr-dr:+.4f}")

    print(f"\n=== Timing ===")
    t = report['timing']
    print(f"  Nav: {t['nav_avg_ms']:.1f}ms  Dense: {t['dense_avg_ms']:.1f}ms  Speedup: {t['speedup']:.1f}x")

---
## Experiment 2: Eval Spreading (EXP-2.2)

Tests whether spreading activation from proof history improves premise retrieval.

**Gate:** ≥5% recall@16 improvement at proof step 3+

In [ ]:
!PYTHONPATH=/content/wayfinder python scripts/eval_spreading.py \
    --config configs/wayfinder.yaml \
    --checkpoint models/NAV-002_step5000.pt \
    --device cuda \
    --samples 500 \
    --output runs/eval_spreading_v2_gpu.json

In [ ]:
import json, os
result_path = 'runs/eval_spreading_v2_gpu.json'
if not os.path.exists(result_path):
    print(f'ERROR: {result_path} not found. Check the eval_spreading output above for errors.')
else:
    with open(result_path) as f:
        spread_report = json.load(f)
    print(json.dumps(spread_report, indent=2))

---
## Experiment 3: Anchor Gap Analysis Re-run (EXP-0.2)

Re-validates anchor gap analysis on the expanded 242K entity DB.
The original pass (recall@16=100%) was on the 78K trace-bounded DB.

**Gate:** recall@16 ≥ 70% on perfect queries

In [ ]:
!PYTHONPATH=/content/wayfinder python scripts/anchor_gap_analysis.py \
    --db data/proof_network.db \
    --samples 500 \
    --output runs/anchor_gap_v2.jsonl

---
## Download Results

Download the results JSON files for recording in EXPERIMENT_RESULTS.md

In [ ]:
import shutil

# Collect all result files
result_files = [
    'runs/eval_retrieval_v2_gpu.json',
    'runs/eval_spreading_v2_gpu.json',
    'runs/anchor_gap_v2.jsonl',
]

os.makedirs('/content/results', exist_ok=True)
for f in result_files:
    if os.path.exists(f):
        shutil.copy(f, f'/content/results/{os.path.basename(f)}')
        print(f'  ✓ {f}')
    else:
        print(f'  ✗ {f} (not found)')

# Zip for download
shutil.make_archive('/content/wayfinder_eval_results', 'zip', '/content/results')
print('\nDownload: /content/wayfinder_eval_results.zip')

# Auto-download in Colab
try:
    from google.colab import files
    files.download('/content/wayfinder_eval_results.zip')
except ImportError:
    print('Not in Colab — download manually from the file browser')